In [82]:
import pandas as pd
from dowhy import CausalModel

#Posição de cada variável para fatiar a database. Nota: o -1 serve para ajustar aos índices do Python começarem em 0
col_tratamento = (194-1, 194)
col_resultado = (117-1, 117)
col_idade = (1971-1, 1972)
col_sexo = (120-1, 120)
col_imc = (1988-1, 1991)
col_pressao = (1896-1, 1896)
col_colesterol = (1898-1, 1898)
col_ativ_fisica = (227-1, 227)
col_educacao = (158-1, 158)
col_renda = (175-1, 176)

In [83]:
#Listas que armazenarão os dados
tratamento = []
resultado = []
idade = []
sexo = []
imc = []
pressao = []
colesterol = []
ativ_fisica = []
educacao = []
renda = []

invalidos1 = {'7', '9', ' '} #Set que verifica se é preciso ignorar a entrada com base nas variáveis tratamento, resultado, pressão alta, colesterol alto, atividade física ou educação
invalidos2 = {'77', '99', ' '} #Set que verifica se é preciso ignorar a entrada com base na variável renda

In [84]:
#Salvando as informações da base de dados no Python
with open('LLCP2015.ASC', mode='rt', encoding='utf-8') as db:
    for linha in db:
        #Recolhimento dos dados através do fatiamento da coluna. São marcadas como temporária pois serão tratadas e limpas depois
        temp_tratamento = linha[col_tratamento[0]:col_tratamento[1]]
        temp_resultado = linha[col_resultado[0]:col_resultado[1]]
        temp_idade = linha[col_idade[0]:col_idade[1]]
        temp_sexo = linha[col_sexo[0]:col_sexo[1]]
        temp_imc = linha[col_imc[0]:col_imc[1]]
        temp_pressao = linha[col_pressao[0]:col_pressao[1]]
        temp_colesterol = linha[col_colesterol[0]:col_colesterol[1]]
        temp_ativ_fisica = linha[col_ativ_fisica[0]:col_ativ_fisica[1]]
        temp_educacao = linha[col_educacao[0]:col_educacao[1]]
        temp_renda = linha[col_renda[0]:col_renda[1]]
                
        #Testes de verificação se há alguma entrada inválida
        if any(dado in invalidos1 for dado in [temp_tratamento, temp_resultado, temp_pressao, temp_colesterol, temp_ativ_fisica, temp_educacao]):
            continue
        elif temp_idade == '14':
            continue
        elif temp_imc == '    ':
            continue
        elif temp_renda in invalidos2:
            continue
        
        #Correção e ajuste dos dados que precisam
        dado_tratamento = 1 if temp_tratamento == '1' else 0
        dado_resultado = 1 if temp_resultado == '1' else 0
        dado_sexo = 1 if temp_sexo == '1' else 0
        dado_imc = int(temp_imc)/100
        dado_pressao = 0 if temp_pressao == '1' else 1
        dado_colesterol = 0 if temp_colesterol == '1' else 1
        dado_ativ_fisica = 1 if temp_ativ_fisica == '1' else 0
        
        tratamento.append(dado_tratamento)
        resultado.append(dado_resultado)
        idade.append(int(temp_idade))
        sexo.append(dado_sexo)
        imc.append(dado_imc)
        pressao.append(dado_pressao)
        colesterol.append(dado_colesterol)
        ativ_fisica.append(dado_ativ_fisica)
        educacao.append(int(temp_educacao))
        renda.append(int(temp_renda))

In [ ]:
df = pd.DataFrame({
    'Fumante':tratamento,
    'Diabetico':resultado,
    'Idade':idade,
    'Sexo':sexo,
    'IMC':imc,
    'Pressao_alta':pressao,
    'Colesterol_alto':colesterol,
    'Atividade_fisica':ativ_fisica,
    'Educacao':educacao,
    'Renda':renda
})

In [85]:
df['Fumante'] = pd.to_numeric(df['Fumante']).astype(int)
df['Diabetico'] = pd.to_numeric(df['Diabetico']).astype(int)

In [86]:
model = CausalModel(
    data=df,
    treatment='Fumante', # Variável de tratamento (Fumante ou não)
    outcome='Diabetico',    # Variável de resultado (Diabetes)
    common_causes=['Idade', 'Sexo', 'IMC', 'Pressao_alta', 'Atividade_fisica', 'Educacao', 'Renda'] # Confounding variables
)

In [87]:
identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)

In [88]:
estimate = model.estimate_effect(
    identified_estimand,
    method_name='backdoor.propensity_score_matching',
    target_units='ate',
    method_params={
        'distance_metric': 'minkowski',
        'p': 2,
        'caliper': 0.01
    }
)

propensity_score_matching


D:\Programming_and_Scripts\replicacao_de_artigos_causalidade\Tabagismo e Diabetes\venv\Lib\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [89]:
print(f"ATE Estimado: {estimate.value}")

ATE Estimado: 0.004953927397481723
